In [1]:
# Cell 1: Setup & Constants
# Notebook 06: Dim_Product — Gold_SalesOps_Dim_Product
# Source: Product (Silver)
# Grain: ProductId (one row per product — confirmed unique)

from pyspark.sql import functions as F

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")

StatementMeta(, 3e955b50-e657-4021-a980-a443d31b0284, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo


In [2]:
# Cell 2: Load Product table

df_product = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Product")
    .filter(F.col("IsDeleted") == False)
)

total = df_product.count()
distinct = df_product.select("ProductId").distinct().count()

print(f"Product rows:        {total:,}")
print(f"Distinct ProductId:  {distinct:,}")
print(f"ProductId unique?    {'YES' if distinct == total else 'NO — DUPLICATES EXIST'}")

StatementMeta(, 3e955b50-e657-4021-a980-a443d31b0284, 4, Finished, Available, Finished, False)

Product rows:        22,051
Distinct ProductId:  22,051
ProductId unique?    YES


In [3]:
# Cell 3: Select Final Columns

dim_product = df_product.select(
    # Key
    "ProductId",
    "SourceId",
    "ProductKey",
    # Product Details
    "Product",
    "ProductDescription",
    # Global Product Hierarchy
    "GlobalProductClassId",
    "GlobalProductClass",
    "GlobalProductLineId",
    "GlobalProductLine",
    "GlobalProductId",
    "GlobalProduct",
)

row_count = dim_product.count()
print(f"Dim_Product rows: {row_count:,}")

# NULL counts for key columns
print("\nKey column NULL counts:")
print("-" * 55)
for col_name in ["ProductKey", "Product", "GlobalProductClassId", "GlobalProductClass", "GlobalProductLineId", "GlobalProductLine", "GlobalProductId", "GlobalProduct"]:
    null_count = dim_product.filter(F.col(col_name).isNull()).count()
    pct = null_count / row_count * 100 if row_count > 0 else 0
    print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")

StatementMeta(, 3e955b50-e657-4021-a980-a443d31b0284, 5, Finished, Available, Finished, False)

Dim_Product rows: 22,051

Key column NULL counts:
-------------------------------------------------------
  ProductKey                                     0  (  0.0%)
  Product                                      332  (  1.5%)
  GlobalProductClassId                       9,702  ( 44.0%)
  GlobalProductClass                         9,702  ( 44.0%)
  GlobalProductLineId                        9,702  ( 44.0%)
  GlobalProductLine                          9,738  ( 44.2%)
  GlobalProductId                            9,702  ( 44.0%)
  GlobalProduct                              9,702  ( 44.0%)


In [4]:
# Cell 4: Write to Gold Lakehouse

dim_product.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("Gold_SalesOps_Dim_Product")

final_count = spark.read.table("Gold_SalesOps_Dim_Product").count()
print(f"Gold_SalesOps_Dim_Product written: {final_count:,} rows")

StatementMeta(, 3e955b50-e657-4021-a980-a443d31b0284, 6, Finished, Available, Finished, False)

Gold_SalesOps_Dim_Product written: 22,051 rows
